<a href="https://colab.research.google.com/github/saileepanchbhai/Advance-Machine-Learning-Lab/blob/main/Boosting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import Required Libraries

import numpy as np
import pandas as pd

# For splitting dataset and cross validation
from sklearn.model_selection import train_test_split, KFold, cross_val_score

# Base learner
from sklearn.tree import DecisionTreeClassifier

# Boosting algorithm
from sklearn.ensemble import AdaBoostClassifier

# Evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

Saving diabetes.csv to diabetes.csv
User uploaded file "diabetes.csv" with length 23873 bytes


In [4]:
df = pd.read_csv("diabetes.csv")
print("First 5 rows of dataset:")
print(df.head())

First 5 rows of dataset:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


In [5]:
# Separate features (X) and target (y)
X = df.drop("Outcome", axis=1)   # Independent variables
y = df["Outcome"]                # Target variable (0 = No Diabetes, 1 = Diabetes)

In [6]:
# Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,      # 30% for testing
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 537
Testing samples: 231


In [8]:
# ==============================
# Base Model: Decision Tree
# ==============================

# Create Decision Tree (weak learner)
dt_model = DecisionTreeClassifier(max_depth=1)

# Train model
dt_model.fit(X_train, y_train)

# Make predictions
y_pred_dt = dt_model.predict(X_test)

# Evaluate accuracy
dt_accuracy = accuracy_score(y_test, y_pred_dt)

print("\nDecision Tree Accuracy:", dt_accuracy)


Decision Tree Accuracy: 0.7186147186147186


In [9]:
# ==============================
# AdaBoost Model
# ==============================

# Create AdaBoost classifier
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # Weak learner
    n_estimators=50,       # Number of boosting rounds
    learning_rate=1,
    random_state=42
)

# Train AdaBoost model
ada_model.fit(X_train, y_train)

# Make predictions
y_pred_ada = ada_model.predict(X_test)

# Evaluate accuracy
ada_accuracy = accuracy_score(y_test, y_pred_ada)

print("AdaBoost Accuracy:", ada_accuracy)

AdaBoost Accuracy: 0.7532467532467533


In [10]:
# ==============================
# Detailed Evaluation
# ==============================

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_ada))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_ada))


Confusion Matrix:
[[124  27]
 [ 30  50]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.82      0.81       151
           1       0.65      0.62      0.64        80

    accuracy                           0.75       231
   macro avg       0.73      0.72      0.73       231
weighted avg       0.75      0.75      0.75       231



In [16]:
# ==============================
# K-Fold Cross Validation
# ==============================

# Define K-Fold (5 splits)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Perform cross validation on AdaBoost model
cv_scores = cross_val_score(ada_model, X, y, cv=kfold)

print("\nCross Validation Scores:", cv_scores)
print("Mean CV Accuracy:", np.mean(cv_scores))


Cross Validation Scores: [0.77922078 0.77272727 0.73376623 0.78431373 0.71895425]
Mean CV Accuracy: 0.757796451914099


In [17]:
print("\n========== MODEL ACCURACY COMPARISON ==========\n")

print("1️. Decision Tree Accuracy (Test Set):",
      round(dt_accuracy * 100, 2), "%")

print("2️. AdaBoost Accuracy (Test Set):",
      round(ada_accuracy * 100, 2), "%")

print("3️. AdaBoost Mean Accuracy (5-Fold CV):",
      round(cv_mean_accuracy * 100, 2), "%")



========== MODEL ACCURACY COMPARISON ==========

1️. Decision Tree Accuracy (Test Set): 71.86 %
2️. AdaBoost Accuracy (Test Set): 75.32 %
3️. AdaBoost Mean Accuracy (5-Fold CV): 75.78 %
